In [7]:
import duckdb
import time
import os

task_events_file = "/media/mudda/extra/merged_task_events.csv"
task_usage_file = "/media/mudda/extra/task_usages_?/merged_task_usage.csv"
output_csv = "/media/mudda/extra/task_usages_?/training_dataset.csv"
temp_dir = "/media/mudda/extra/tmp"

print("Starting FULL merge with all columns (CSV output)...")
start_time = time.time()

os.makedirs(temp_dir, exist_ok=True)

con = duckdb.connect("training_merge.db")

# Safe settings for 12GB RAM
con.execute("PRAGMA memory_limit='8GB';")
con.execute("PRAGMA threads=4;")
con.execute(f"PRAGMA temp_directory='{temp_dir}';")
con.execute("PRAGMA enable_progress_bar;")

con.execute(f"""
COPY (
    WITH task_labels AS (
        SELECT 
            job_id,
            task_index,
            MAX(CASE WHEN event_type = 3 THEN 1 ELSE 0 END) AS failed
        FROM read_csv_auto('{task_events_file}', ignore_errors=true)
        GROUP BY job_id, task_index
    ),
    
    events_deduplicated AS (
        SELECT e.*, l.failed
        FROM read_csv_auto('{task_events_file}', ignore_errors=true) e
        INNER JOIN task_labels l
        ON e.job_id = l.job_id
        AND e.task_index = l.task_index
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY e.job_id, e.task_index
            ORDER BY e.time DESC
        ) = 1
    )
    
    SELECT 
        u.*,
        e.*
    FROM read_csv_auto('{task_usage_file}', ignore_errors=true) u
    INNER JOIN events_deduplicated e
    ON u.job_id = e.job_id
    AND u.task_index = e.task_index
)
TO '{output_csv}'
(HEADER, DELIMITER ',');
""")

total_time = time.time() - start_time

print("\n" + "="*50)
print("✅ FULL CSV TRAINING DATASET CREATED")
print(f"⏱ Total time: {total_time/60:.2f} minutes")

if os.path.exists(output_csv):
    size_gb = os.path.getsize(output_csv) / (1024**3)
    print(f"📦 CSV file size: {size_gb:.2f} GB")

con.close()

Starting FULL merge with all columns (CSV output)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


✅ FULL CSV TRAINING DATASET CREATED
⏱ Total time: 4.63 minutes
📦 CSV file size: 62.90 GB


In [ ]:
import pandas as pd

# Load datasets
events = pd.read_csv('/media/mudda/extra/merged_task_events.csv')
usage = pd.read_csv('/media/mudda/extra/cluster/merged_task_usage.csv')

# 1. Extract only the failure events (event_type 3 is 'FAIL')
failures = events[events['event_type'] == 3][['job_id', 'task_index', 'time']]
failures = failures.rename(columns={'time': 'failure_timestamp'})

# 2. Merge failure timestamps into the usage telemetry
# This gives every usage row a "future outcome"
merged_df = pd.merge(usage, failures, on=['job_id', 'task_index'], how='left')

# 3. Create Target Labels
# will_fail: 1 if this task eventually crashes, 0 otherwise
merged_df['will_fail'] = merged_df['failure_timestamp'].notna().astype(int)

# ttf: Time remaining until the crash (in the dataset's time units)
merged_df['ttf'] = merged_df['failure_timestamp'] - merged_df['end_time']

# 4. Remove usage records that occurred AFTER the failure (if any)
merged_df = merged_df[~(merged_df['ttf'] < 0)]

# 5. Save the final training dataset
merged_df.to_csv('failure_prediction_dataset.csv', index=False)

In [1]:
import pandas as pd
from tqdm import tqdm
import os

events_path = "/media/mudda/extra/merged_task_events.csv"
usage_path = "/media/mudda/extra/cluster/merged_task_usage.csv"
output_path = "failure_prediction_dataset.csv"

CHUNK_SIZE = 500_000   # Adjust if needed (lower if RAM pressure)

print("Step 1: Loading failure events...")

# -------------------------------------------------
# 1. Load ONLY failure events into memory
# -------------------------------------------------
failures = pd.read_csv(
    events_path,
    usecols=["job_id", "task_index", "event_type", "time"]
)

failures = failures[failures["event_type"] == 3][
    ["job_id", "task_index", "time"]
].rename(columns={"time": "failure_timestamp"})

print(f"Total failure records: {len(failures)}")

# Convert to dictionary for ultra-fast lookup
failure_lookup = failures.set_index(
    ["job_id", "task_index"]
)["failure_timestamp"].to_dict()

del failures  # free memory

print("Failure lookup table created.\n")

# -------------------------------------------------
# 2. Process usage file in chunks
# -------------------------------------------------

total_lines = sum(1 for _ in open(usage_path)) - 1
total_chunks = total_lines // CHUNK_SIZE + 1

print("Step 2: Processing usage data in chunks...")
print(f"Total rows in usage file: {total_lines}")
print(f"Processing in {total_chunks} chunks...\n")

first_chunk = True

for chunk in tqdm(
    pd.read_csv(usage_path, chunksize=CHUNK_SIZE),
    total=total_chunks,
    desc="Processing"
):
    # Merge logic (manual, memory-safe)
    chunk["failure_timestamp"] = chunk.set_index(
        ["job_id", "task_index"]
    ).index.map(failure_lookup)

    # Create will_fail
    chunk["will_fail"] = chunk["failure_timestamp"].notna().astype(int)

    # Create ttf
    chunk["ttf"] = chunk["failure_timestamp"] - chunk["end_time"]

    # Remove records after failure
    chunk = chunk[~(chunk["ttf"] < 0)]

    # Append to CSV safely
    chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

print("\n✅ Dataset creation completed successfully.")

Step 1: Loading failure events...
Total failure records: 14121
Failure lookup table created.

Step 2: Processing usage data in chunks...
Total rows in usage file: 27646684
Processing in 56 chunks...



Processing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 56/56 [03:17<00:00,  3.53s/it]


✅ Dataset creation completed successfully.


In [ ]:
import pandas as pd
from tqdm import tqdm

events_path = "/Downloads/merged_task_events.csv"
usage_path = "/media/mudda/extra/task_usages_?/merged_task_usage.csv"
output_path = "/media/mudda/extra/failure_prediction_dataset.csv"

CHUNK_SIZE = 500_000

print("Step 1: Loading failure events...")

# -------------------------------------------------
# 1. Load ONLY failure events (memory optimized)
# -------------------------------------------------
failures = pd.read_csv(
    events_path,
    usecols=["job_id", "task_index", "event_type", "time"],
    dtype={
        "job_id": "int64",
        "task_index": "int32",
        "event_type": "int8",
        "time": "int64"
    }
)

failures = failures[failures["event_type"] == 3][
    ["job_id", "task_index", "time"]
].rename(columns={"time": "failure_timestamp"})

print(f"Total failure records: {len(failures)}")

# Dictionary lookup (very memory efficient)
failure_lookup = failures.set_index(
    ["job_id", "task_index"]
)["failure_timestamp"].to_dict()

del failures  # free RAM

print("Failure lookup table created.\n")

# -------------------------------------------------
# 2. Process usage file in chunks
# -------------------------------------------------

total_lines = sum(1 for _ in open(usage_path)) - 1
total_chunks = total_lines // CHUNK_SIZE + 1

print("Step 2: Processing usage data in chunks...")
print(f"Total rows in usage file: {total_lines}")
print(f"Processing in {total_chunks} chunks...\n")

first_chunk = True

for chunk in tqdm(
    pd.read_csv(
        usage_path,
        chunksize=CHUNK_SIZE,
        dtype={
            "job_id": "int64",
            "task_index": "int32",
            "end_time": "int64"
        }
    ),
    total=total_chunks,
    desc="Processing"
):

    # Add failure timestamp using index map (no merge = less RAM)
    idx = chunk.set_index(["job_id", "task_index"]).index
    chunk["failure_timestamp"] = idx.map(failure_lookup)

    # Create failed column (0 or 1)
    chunk["failed"] = chunk["failure_timestamp"].notna().astype("int8")

    # Time-to-failure
    chunk["ttf"] = chunk["failure_timestamp"] - chunk["end_time"]

    # Remove records after failure
    chunk = chunk[~(chunk["ttf"] < 0)]

    # Write safely
    chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    # Free memory immediately
    del chunk

print("\n✅ Dataset creation completed successfully.")

In [1]:
import pandas as pd
from tqdm import tqdm

# ==============================
# PATHS
# ==============================

events_path = "/home/mudda/Downloads/0to150_merged_task_events.csv"
usage_path = "/media/mudda/extra/task_usages_?/merged_task_usage.csv"
output_path = "/media/mudda/extra/final_failure_prediction_dataset.csv"

CHUNK_SIZE = 100_000
PRED_WINDOW = 5_000_000   # adjust prediction window

# ==============================
# STEP 1: LOAD EVENTS (OPTIMIZED)
# ==============================

print("Loading task events...")

events = pd.read_csv(
    events_path,
    usecols=[
        "job_id","task_index","time","event_type",
        "priority","scheduling_class",
        "cpu_request","memory_request","disk_request"
    ],
    dtype={
        "job_id": "int64",
        "task_index": "int32",
        "time": "int64",
        "event_type": "int8",
        "priority": "int32",
        "scheduling_class": "int8",
        "cpu_request": "float32",
        "memory_request": "float32",
        "disk_request": "float32"
    }
)

# ==============================
# STEP 2: FAILURE TIME
# ==============================

failures = events[events["event_type"] == 3][
    ["job_id","task_index","time"]
].rename(columns={"time":"failure_time"})

failure_lookup = failures.set_index(
    ["job_id","task_index"]
)["failure_time"].to_dict()

print("Failure lookup created.")

# ==============================
# STEP 3: STATIC TASK FEATURES
# ==============================

static_features = (
    events.sort_values("time")
    .drop_duplicates(["job_id","task_index"], keep="first")
    [[
        "job_id","task_index",
        "priority","scheduling_class",
        "cpu_request","memory_request","disk_request"
    ]]
)

static_lookup = static_features.set_index(
    ["job_id","task_index"]
).to_dict("index")

print("Static feature lookup created.")

del events, failures, static_features

# ==============================
# STEP 4: PROCESS USAGE IN CHUNKS
# ==============================

print("\nProcessing usage file in chunks...\n")

first_chunk = True

for chunk in tqdm(
    pd.read_csv(
        usage_path,
        chunksize=CHUNK_SIZE,
        dtype={
            "job_id": "int64",
            "task_index": "int32",
            "start_time": "int64",
            "end_time": "int64",
            "cpu_rate": "float32",
            "canonical_memory_usage": "float32",
            "maximum_cpu_rate": "float32",
            "disk_io_time": "float32",
            "cycles_per_instruction": "float32"
        }
    ),
    desc="Processing"
):

    # ------------------------------
    # FAILURE TIME
    # ------------------------------

    idx = chunk.set_index(["job_id","task_index"]).index

    chunk["failure_time"] = idx.map(failure_lookup)

    chunk["failed"] = chunk["failure_time"].notna().astype("int8")

    # ------------------------------
    # REMOVE DATA AFTER FAILURE
    # ------------------------------

    chunk = chunk[
        (chunk["failed"] == 0) |
        (chunk["end_time"] <= chunk["failure_time"])
    ]

    # ------------------------------
    # EARLY FAILURE TARGET
    # ------------------------------

    chunk["will_fail_soon"] = (
        (chunk["failed"] == 1) &
        ((chunk["failure_time"] - chunk["end_time"]) <= PRED_WINDOW)
    ).astype("int8")

    # ------------------------------
    # ADD STATIC FEATURES
    # ------------------------------

    static_data = idx.map(static_lookup)

    static_df = pd.DataFrame(list(static_data))

    chunk[[
        "priority","scheduling_class",
        "cpu_request","memory_request","disk_request"
    ]] = static_df

    # ------------------------------
    # BEHAVIOR FEATURES
    # ------------------------------

    chunk["cpu_pressure"] = (
        chunk["cpu_rate"] /
        (chunk["cpu_request"] + 1e-5)
    )

    chunk["memory_pressure"] = (
        chunk["canonical_memory_usage"] /
        (chunk["memory_request"] + 1e-5)
    )

    chunk["cpu_spike"] = (
        chunk["maximum_cpu_rate"] -
        chunk["cpu_rate"]
    )

    chunk["resource_pressure"] = (
        chunk["cpu_rate"] *
        chunk["canonical_memory_usage"]
    )

    # ------------------------------
    # SAVE SAFELY
    # ------------------------------

    chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False
    del chunk

print("\n✅ Final dataset created successfully.")

Loading task events...
Failure lookup created.
Static feature lookup created.

Processing usage file in chunks...



Processing: 2507it [3:54:51,  5.62s/it]


✅ Final dataset created successfully.
